# Stage 0 — Environment Setup

Goal: a reproducible Python environment and a verified Copernicus Marine login. Do this **once per machine**. If you've already done it for another project, skip to the verification cells.

## What we need
- Python 3.11+
- A virtual environment (`.venv`) at the project root
- The packages listed below
- A `copernicusmarine login` token cached in your home dir
- Network access to the CMEMS catalogue + ECMWF CDS (later, for ERA5)

## 0.1 — Create the venv & install packages

**Run these in a terminal, not in this notebook** (a notebook can't activate its own venv). PowerShell on Windows:

```powershell
cd c:\GIT\mcast.pages.ai.v2
python -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install "copernicusmarine>=2.4.1" xarray netCDF4 dask zarr `
            numpy pandas scikit-learn xgboost lightgbm `
            matplotlib cartopy shap cdsapi tqdm pyarrow seaborn
```

Then **point Jupyter / VS Code at the venv kernel** (`.venv\Scripts\python.exe`) before running the verification cells below.

## 0.2 — One-time Copernicus Marine login

Register free at <https://data.marine.copernicus.eu> if you don't already have an account. Then in a terminal (with the venv active):

```powershell
copernicusmarine login
```

You'll be prompted for username + password. The token is cached in your home directory (`~/.copernicusmarine/`) so future API calls authenticate silently.

## 0.3 — Verify everything imports

If any of these fail with `ModuleNotFoundError`, the kernel isn't pointing at the venv (or pip install missed a package). Fix that before continuing.

In [ ]:
import sys
import importlib

REQUIRED = [
    "copernicusmarine", "xarray", "netCDF4", "dask", "zarr",
    "numpy", "pandas", "sklearn", "xgboost", "lightgbm",
    "matplotlib", "shap", "cdsapi", "tqdm", "pyarrow", "seaborn",
]

print(f"Python {sys.version.split()[0]} @ {sys.executable}\n")

missing = []
for pkg in REQUIRED:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  OK  {pkg:20s} {ver}")
    except Exception as e:
        print(f"MISS  {pkg:20s} {e}")
        missing.append(pkg)

if missing:
    print(f"\nMissing: {missing} — pip install them into the active venv.")
else:
    print("\nAll required packages present.")

## 0.4 — Verify CMEMS catalogue access

`describe()` is a metadata-only call. It fails fast on auth errors (re-run `copernicusmarine login`) and on network/firewall issues — so it's a clean separator between "environment broken" and "data broken".

In [ ]:
import copernicusmarine

DATASETS_TO_PROBE = [
    "cmems_obs-oc_med_bgc-plankton_my_l3-multi-1km_P1D",   # Stage 1 target
]

for ds_id in DATASETS_TO_PROBE:
    try:
        copernicusmarine.describe(dataset_id=ds_id)
        print(f"  OK  {ds_id}")
    except Exception as e:
        print(f"FAIL  {ds_id}\n      {e}")

---
## Validation Gate 0

Three things to confirm before you move on:

1. All required packages imported without error (cell 0.3).
2. `describe()` on the Stage 1 dataset returned without error (cell 0.4) — proves both auth and network.
3. You understand **why** we picked `*_my_l3-multi-1km_P1D`:
   - `l3` → not gap-filled (we want honest NaNs as the methodological core)
   - `multi` → multi-sensor merge, maximises daily coverage without temporal interpolation
   - `my` → multi-year reanalysis, stable historical record from 1997 (use the `nrt` twin for operational T+1…T+7 inference later)
   - `1km` → finest grid available for the Med; everything else gets regridded *onto* this
   - `P1D` → daily cadence

If failing: auth errors → re-run `copernicusmarine login` in a terminal. Network → check firewall/proxy. Dataset-not-found → CMEMS may have versioned the ID; run `copernicusmarine describe --contains "med_bgc-plankton"` to list current short-names and update `DATASETS_TO_PROBE`.

---
## Next: `01-Data-Acquisition.ipynb`

Pulls the L3 CHL cube for the Malta–Sicily box (3 years to start) and runs Validation Gate 1.